# Coding Exercise: Diagnose a Problematic ML Pipeline

In this exercise, you'll be implementing the simple bias identification methodologies we identified in this lesson towards the German Credit Dataset, offered by the AIF360 toolkit (under [Apache License 2.0](https://www.apache.org/licenses/LICENSE-2.0)).

Here is some context on the dataset:

The German Credit Dataset, or German Dataset, has 20 features and 1000 instances. The target variable for this dataset is a binary label of good or bad credit risks. Variables include credit history, credit amount, sex, age, and others.

For this analysis, we use all of the variables in the dataset.

Dataset citation: Hofmann, H. (1994). UCI Machine Learning Repository [https://archive.ics.uci.edu/ml/datasets/statlog+(german+credit+data)]. Irvine, CA: University of California, School of Information and Computer Science.

# Diagnose a Problematic ML Pipeline

In this exercise, you will implement a series of steps towards diagnosing bias in an ML pipeline with the bias identification methdologies we identified, using the German Credit Dataset provided by the AIF360 toolkit. We have included a few helper code blocks for loading the dataset and generating fairness metrics from [the AIF360 tutorials](https://github.com/Trusted-AI/AIF360/blob/master/examples/tutorial_credit_scoring.ipynb) that you will be building on for this simple analysis.

In [ ]:
# Load all necessary packages
import sys
sys.path.insert(1, "../../")

import numpy as np
np.random.seed(0)

from aif360.datasets import GermanDataset
from aif360.metrics import BinaryLabelDatasetMetric
from aif360.algorithms.preprocessing import Reweighing

from IPython.display import Markdown, display
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

## Load Data

Below, we load and setup the data, defining our privileged class as male and our label and protected attribute maps.

In [ ]:
#Helper code

#Define the label map for the dataset
label_map = {1.0: 'Good Credit', 2.0: 'Bad Credit'}

#Define the protected attribute for the dataset
protected_attribute_maps = [{1.0: 'Male', 0.0: 'Female'}]

#Load data, defining priviledge class as male
gd = GermanDataset(protected_attribute_names=['sex'],
privileged_classes=[['male']], metadata={'label_map': label_map,
'protected_attribute_maps': protected_attribute_maps})

df, df_dict = gd.convert_to_dataframe()
viz_df = df.copy()

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve
import aif360

# Download the German Credit files expected by AIF360, if they are missing.
german_dir = Path(aif360.__file__).parent / 'data' / 'raw' / 'german'
german_dir.mkdir(parents=True, exist_ok=True)
base_url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/statlog/german/'
for filename in ('german.data', 'german.doc'):
    destination = german_dir / filename
    if not destination.exists():
        urlretrieve(base_url + filename, destination)

print(f'German Credit dataset ready at: {german_dir}')

In [ ]:
import numpy as np
import pandas as pd
from aif360.datasets import GermanDataset

label_map = {1.0: 'Good Credit', 2.0: 'Bad Credit'}
protected_attribute_maps = [{1.0: 'Male', 0.0: 'Female'}]

# AIF360 0.6.1 calls np.issubdtype on pandas StringDtype.
# Use a temporary compatibility wrapper for that check.
_original_issubdtype = np.issubdtype
def _aif360_issubdtype(dtype, kind):
    try:
        return _original_issubdtype(dtype, kind)
    except TypeError:
        return pd.api.types.is_numeric_dtype(dtype) if kind is np.number else False

try:
    np.issubdtype = _aif360_issubdtype
    gd = GermanDataset(
        protected_attribute_names=['sex'],
        privileged_classes=[['male']],
        metadata={
            'label_map': label_map,
            'protected_attribute_maps': protected_attribute_maps
        }
    )
finally:
    np.issubdtype = _original_issubdtype

df, df_dict = gd.convert_to_dataframe()
viz_df = df.copy()

In [ ]:
#Visualize the dataframe
viz_df.head()

## Analyze the representation of the dataset

Using `viz_df`, create two bar plots showing the representation of gender and credit in the dataset. Briefly interpret your findings - which categories seem to have a higher number of instances?

In [ ]:
import matplotlib.pyplot as plt

gender_counts = viz_df['sex'].map({1.0: 'Male', 0.0: 'Female'}).value_counts().reindex(['Male', 'Female'])
credit_counts = viz_df['credit'].map({1.0: 'Good Credit', 2.0: 'Bad Credit'}).value_counts().reindex(['Good Credit', 'Bad Credit'])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

gender_counts.plot(kind='bar', ax=axes[0], color=['steelblue', 'lightcoral'])
axes[0].set_title('Gender Representation')
axes[0].set_xlabel('Gender')
axes[0].set_ylabel('Number of instances')
axes[0].tick_params(axis='x', rotation=0)

credit_counts.plot(kind='bar', ax=axes[1], color=['seagreen', 'goldenrod'])
axes[1].set_title('Credit Representation')
axes[1].set_xlabel('Credit risk')
axes[1].set_ylabel('Number of instances')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

print('Gender counts:\n', gender_counts)
print('\nCredit counts:\n', credit_counts)

### Interpretation

The dataset contains more male than female instances (690 male versus 310 female). It also contains more good-credit than bad-credit instances (700 good-credit versus 300 bad-credit). Therefore, both gender and credit labels are unevenly represented, with males and good-credit cases forming the majority categories.

Analyze the representation of credit, stratified by gender, in the dataset. You can use the pandas groupby() operation to do so. Add a brief interpretation.

In [ ]:
# Count credit outcomes within each gender group.
gender_credit_counts = (
    viz_df.assign(
        gender=viz_df['sex'].map({1.0: 'Male', 0.0: 'Female'}),
        credit_label=viz_df['credit'].map({1.0: 'Good Credit', 2.0: 'Bad Credit'})
    )
    .groupby(['gender', 'credit_label'])
    .size()
    .unstack(fill_value=0)
    .reindex(index=['Male', 'Female'], columns=['Good Credit', 'Bad Credit'])
)

display(gender_credit_counts)

gender_credit_counts.plot(kind='bar', figsize=(8, 4), color=['seagreen', 'goldenrod'])
plt.title('Credit Representation Stratified by Gender')
plt.xlabel('Gender')
plt.ylabel('Number of instances')
plt.xticks(rotation=0)
plt.legend(title='Credit risk')
plt.tight_layout()
plt.show()

### Interpretation

Both gender groups contain more good-credit than bad-credit instances. Male participants have the largest counts in both categories (499 good credit and 191 bad credit), while female participants have fewer instances overall (201 good credit and 109 bad credit). The groups are therefore uneven in size, and the credit distribution is also imbalanced within each group.

## Correlation Matrix

Create a correlation matrix with the `sex`, `age`, and `credit` variables. Interpret the correlations - do you see any strong relationships.

In [ ]:
correlation_matrix = viz_df[['sex', 'age', 'credit']].corr()
display(correlation_matrix.round(3))

fig, ax = plt.subplots(figsize=(6, 5))
image = ax.imshow(correlation_matrix, cmap='coolwarm', vmin=-1, vmax=1)
ax.set_xticks(range(len(correlation_matrix.columns)), correlation_matrix.columns)
ax.set_yticks(range(len(correlation_matrix.index)), correlation_matrix.index)

for row in range(correlation_matrix.shape[0]):
    for col in range(correlation_matrix.shape[1]):
        ax.text(col, row, f'{correlation_matrix.iloc[row, col]:.2f}',
                ha='center', va='center', color='black')

ax.set_title('Correlation Matrix: Sex, Age, and Credit')
fig.colorbar(image, ax=ax, label='Correlation coefficient')
plt.tight_layout()
plt.show()

### Interpretation

The correlations are weak: sex and age have a small positive correlation (approximately 0.16), while credit has weak negative correlations with sex (approximately -0.08) and age (approximately -0.09). No strong linear relationships are apparent among these variables. Because `sex` and `credit` are encoded as numeric labels, these correlations should be treated as descriptive rather than causal.

---
## Confusion Matrix

Next, you'll be asked to create and train a simple Logistic Regression model on the dataset. We've created the training and test data split and applied scaling to the dataset for you.

In [ ]:
#Split the dataset into training and test datasets
dataset_orig_train, dataset_orig_test = gd.split([0.7], shuffle=True)

#Define priviledge and unprivileged groups
privileged_groups = [{'sex': 1}]
unprivileged_groups = [{'sex': 0}]

#Apply scaling to the dataset
scale_orig = StandardScaler()
X_train = scale_orig.fit_transform(dataset_orig_train.features)
y_train = dataset_orig_train.labels.ravel()
w_train = dataset_orig_train.instance_weights.ravel()

Implement and train a simpler logistic regression sklearn model (with all defaults). Generate predictions on the training dataset.

In [ ]:
# Train a default logistic regression model on the scaled training data.
LR_model = LogisticRegression()
LR_model.fit(X_train, y_train)

# Generate predictions for the training dataset.
y_train_pred = LR_model.predict(X_train)

print(f'Training accuracy: {accuracy_score(y_train, y_train_pred):.3f}')

Plot a confusion matrix, and intepret the failures of the model. Which is higher for the model - false negatives or false positives?

In [ ]:
# GermanDataset uses 1 = Good Credit and 2 = Bad Credit.
cm = confusion_matrix(y_train, y_train_pred, labels=[1.0, 2.0])

ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=['Good Credit', 'Bad Credit']
).plot(cmap='Blues')
plt.title('Training Confusion Matrix')
plt.show()

# Treat Bad Credit as the positive/risk class.
false_positives = cm[0, 1]  # Good credit predicted as bad
false_negatives = cm[1, 0]  # Bad credit predicted as good
print(f'False positives: {false_positives}')
print(f'False negatives: {false_negatives}')
print('False negatives are higher.' if false_negatives > false_positives else 'False positives are higher.')

### Interpretation

Treating Bad Credit as the positive/risk class, the model produces approximately 55 false positives (good-credit applicants predicted as bad) and 90 false negatives (bad-credit applicants predicted as good). False negatives are therefore higher. This means the model more often misses applicants who actually have bad credit than incorrectly flags good-credit applicants as risky.

## Fairness Metrics

Let's conclude our diagnosis of bias in the pipeline with an analysis of statistical parity difference and disparate impact in the dataset. We've included the helper code for you below - interpret the findings.

In [ ]:
# Define privileged and unprivileged groups
privileged_groups = [{'sex': 1}]
unprivileged_groups = [{'sex': 0}]

#Define DatasetMetric class
metric_orig_train = BinaryLabelDatasetMetric(dataset_orig_train,
                                             unprivileged_groups=unprivileged_groups,
                                             privileged_groups=privileged_groups)
display(Markdown("#### Original training dataset"))
#Print statstical parity difference and disparate impact metrics
print("Statistical Parity Difference between unprivileged and privileged groups = %f" % metric_orig_train.mean_difference())
print("Disparate Impact between unprivileged and privileged groups = %f" % metric_orig_train.disparate_impact())

### Interpretation

The statistical parity difference is approximately `-0.108`, meaning the unprivileged group (female applicants) has a good-credit rate about 10.8 percentage points lower than the privileged group (male applicants) in the training data. The disparate impact is approximately `0.853`, meaning the unprivileged group receives the favorable outcome at about 85.3% of the privileged group’s rate. This indicates a modest disparity in favor of male applicants; it is below perfect parity (`1.0`) but above the commonly used `0.80` four-fifths screening threshold. These metrics identify an association in the data, not proof that the model or gender caused the difference.